# <center>Ensemble & Stacking: Win Competitions</center>

<center>

![Python](https://img.shields.io/badge/Python-3.10-blue?logo=python&logoColor=white)
![scikit-learn](https://img.shields.io/badge/scikit--learn-1.3-orange?logo=scikit-learn)
![XGBoost](https://img.shields.io/badge/XGBoost-2.0-green)
![LightGBM](https://img.shields.io/badge/LightGBM-4.1-purple)
![CatBoost](https://img.shields.io/badge/CatBoost-1.2-yellow)
![License](https://img.shields.io/badge/License-MIT-red)

</center>

---

**Author:** Lorenzo Scaturchio  
**Last Updated:** January 2026  
**Kernel Version:** 1.0

> *"The secret to winning Kaggle competitions isn't finding the one perfect model -- it's combining many good models."*  
> -- Every Kaggle Grandmaster, ever.

---

## TL;DR

This notebook is a **complete, practical guide** to ensemble methods -- the single most impactful family of techniques in competitive machine learning. Here's what you'll learn:

| Technique | Key Idea | Difficulty |
|-----------|----------|------------|
| **Bagging** | Reduce variance via bootstrap aggregation | Beginner |
| **Boosting** | Sequentially correct errors | Intermediate |
| **Blending / Weighted Averaging** | Combine predictions with optimal weights | Intermediate |
| **Stacking** | Train a meta-learner on out-of-fold predictions | Advanced |
| **Snapshot Ensembles** | Free ensembles from a single training run | Advanced |
| **Knowledge Distillation** | Compress ensemble into a single model | Advanced |

If you find this notebook useful, please consider giving it an **upvote** -- it helps others discover it!

## Table of Contents

1. [Setup & Imports](#1)
2. [Why Ensembles Win](#2)
3. [Bagging](#3)
4. [Boosting](#4)
5. [Blending & Weighted Averaging](#5)
6. [Stacking](#6)
7. [Advanced: Snapshot Ensembles](#7)
8. [Advanced: Knowledge Distillation](#8)
9. [Practical Competition Pipeline](#9)
10. [Diversity Analysis](#10)
11. [Further Reading](#11)

---

<a id="1"></a>
## 1. Setup & Imports

We load every library we will need throughout this notebook. All of these are available in the default Kaggle kernel environment.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Core
import numpy as np
import pandas as pd
from itertools import combinations

# Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Sklearn - data & metrics
from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import (
    KFold, StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.metrics import (
    accuracy_score, log_loss, roc_auc_score, classification_report
)
from sklearn.preprocessing import StandardScaler

# Sklearn - models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    AdaBoostClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Gradient Boosting Frameworks
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Optimization
from scipy.optimize import minimize

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Plot style
plt.style.use("fivethirtyeight")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

print("All imports successful!")

> **Key Takeaway -- Why this matters:**  
> Ensembling is responsible for virtually every top finish in Kaggle history. The Netflix Prize (2009), the Higgs Boson challenge, Otto Group, and countless others were all won by ensembles. Understanding these techniques is the single highest-ROI skill in competitive ML.

<a id="2"></a>
## 2. Why Ensembles Win

### 2.1 The Bias-Variance Tradeoff

Every model's error can be decomposed into three parts:

$$\text{Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

- **High-bias models** (e.g., logistic regression) underfit -- they miss patterns.
- **High-variance models** (e.g., deep decision trees) overfit -- they memorize noise.

Ensembles attack **both**:
- **Bagging** reduces variance by averaging many high-variance models.
- **Boosting** reduces bias by sequentially correcting errors.
- **Stacking** captures complementary strengths of diverse models.

### 2.2 Wisdom of Crowds

If individual models are (1) **better than random** and (2) **diverse** (i.e., they make different errors), then the majority vote will almost always outperform any individual. This is [Condorcet's jury theorem](https://en.wikipedia.org/wiki/Condorcet%27s_jury_theorem) applied to ML.

In [ ]:
# --- Bias-Variance Tradeoff Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

np.random.seed(SEED)
x_true = np.linspace(0, 2 * np.pi, 200)
y_true = np.sin(x_true)

# High Bias (underfitting)
ax = axes[0]
ax.plot(x_true, y_true, "k-", lw=2, label="True function")
for i in range(5):
    noise = np.random.normal(0, 0.1, len(x_true))
    coef = np.polyfit(x_true, y_true + noise, 1)
    y_pred = np.polyval(coef, x_true)
    ax.plot(x_true, y_pred, alpha=0.5, lw=1)
ax.set_title("High Bias (Linear Fit)", fontsize=14, fontweight="bold")
ax.set_ylim(-2, 2)
ax.legend(fontsize=10)

# High Variance (overfitting)
ax = axes[1]
ax.plot(x_true, y_true, "k-", lw=2, label="True function")
for i in range(5):
    noise = np.random.normal(0, 0.3, len(x_true))
    coef = np.polyfit(x_true, y_true + noise, 15)
    y_pred = np.polyval(coef, x_true)
    ax.plot(x_true, y_pred, alpha=0.5, lw=1)
ax.set_title("High Variance (Degree-15 Poly)", fontsize=14, fontweight="bold")
ax.set_ylim(-2, 2)
ax.legend(fontsize=10)

# Ensemble (just right)
ax = axes[2]
ax.plot(x_true, y_true, "k-", lw=2, label="True function")
preds = []
for i in range(20):
    noise = np.random.normal(0, 0.3, len(x_true))
    coef = np.polyfit(x_true, y_true + noise, 15)
    preds.append(np.polyval(coef, x_true))
ensemble_pred = np.mean(preds, axis=0)
ax.plot(x_true, ensemble_pred, "r-", lw=2.5, label="Ensemble avg (20 models)")
ax.set_title("Ensemble Average", fontsize=14, fontweight="bold")
ax.set_ylim(-2, 2)
ax.legend(fontsize=10)

plt.tight_layout()
plt.suptitle("Why Ensembles Work: Averaging Reduces Variance", y=1.03,
             fontsize=16, fontweight="bold")
plt.show()

In [ ]:
# --- Diversity & Majority Vote Simulation ---
from scipy.stats import binom

n_voters = np.arange(1, 101, 2)  # odd numbers only
accuracies = [0.55, 0.60, 0.65, 0.70, 0.75]

fig, ax = plt.subplots(figsize=(12, 6))
for acc in accuracies:
    ensemble_acc = []
    for n in n_voters:
        majority = n // 2 + 1
        prob = 1 - binom.cdf(majority - 1, n, acc)
        ensemble_acc.append(prob)
    ax.plot(n_voters, ensemble_acc, lw=2, label=f"Individual acc = {acc:.0%}")

ax.axhline(y=1.0, color="gray", ls="--", alpha=0.5)
ax.set_xlabel("Number of Models (voters)", fontsize=13)
ax.set_ylabel("Ensemble Accuracy (majority vote)", fontsize=13)
ax.set_title("Condorcet's Jury Theorem: More Diverse Models = Better Ensemble",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.set_ylim(0.4, 1.05)
plt.tight_layout()
plt.show()

### 2.3 Historical Kaggle Wins Powered by Ensembles

| Competition | Year | Winning Approach |
|-------------|------|------------------|
| Netflix Prize | 2009 | Blend of 800+ models |
| Higgs Boson | 2014 | XGBoost + ensemble of neural nets |
| Otto Group Product Classification | 2015 | 3-level stacking with 30+ base models |
| Bosch Production Line | 2016 | XGBoost + LightGBM ensemble |
| Porto Seguro Safe Driver | 2017 | Stacked GBMs + neural nets |
| Home Credit Default Risk | 2018 | LightGBM stacking with feature engineering |
| IEEE-CIS Fraud Detection | 2019 | Multi-level stacking, rank averaging |
| Jane Street Market Prediction | 2021 | Ensemble of neural nets + GBMs |

> The pattern is clear: **ensembles dominate competitions**.

---

<a id="3"></a>
## 3. Bagging (Bootstrap Aggregating)

**Core idea:** Train multiple models on different bootstrap samples of the data, then aggregate predictions (vote for classification, average for regression).

**Why it works:** Each bootstrap sample leaves out ~36.8% of data (out-of-bag samples). Different training sets lead to different models, and averaging reduces variance.

### 3.1 Bagging from Scratch

In [ ]:
class BaggingFromScratch:
    """Bagging classifier implemented from scratch."""

    def __init__(self, base_estimator, n_estimators=10, sample_ratio=1.0,
                 random_state=42):
        self.base_estimator = base_estimator
        self.n_estimators = n_estimators
        self.sample_ratio = sample_ratio
        self.random_state = random_state
        self.estimators_ = []
        self.oob_indices_ = []

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        n_samples = X.shape[0]
        sample_size = int(n_samples * self.sample_ratio)

        self.estimators_ = []
        self.oob_indices_ = []

        for i in range(self.n_estimators):
            # Bootstrap sample (with replacement)
            indices = rng.choice(n_samples, size=sample_size, replace=True)
            oob = np.setdiff1d(np.arange(n_samples), indices)

            # Clone and fit
            from sklearn.base import clone
            est = clone(self.base_estimator)
            est.fit(X[indices], y[indices])

            self.estimators_.append(est)
            self.oob_indices_.append(oob)

        return self

    def predict(self, X):
        # Majority vote
        predictions = np.array([est.predict(X) for est in self.estimators_])
        from scipy.stats import mode
        majority, _ = mode(predictions, axis=0, keepdims=False)
        return majority.flatten()

    def oob_score(self, X, y):
        """Out-of-bag score estimation."""
        n_samples = X.shape[0]
        oob_preds = np.zeros((n_samples, self.n_estimators)) * np.nan

        for i, (est, oob_idx) in enumerate(
            zip(self.estimators_, self.oob_indices_)
        ):
            if len(oob_idx) > 0:
                oob_preds[oob_idx, i] = est.predict(X[oob_idx])

        # For each sample, take majority vote of estimators that didn't see it
        final_preds = []
        valid_mask = []
        for j in range(n_samples):
            preds_j = oob_preds[j, ~np.isnan(oob_preds[j])]
            if len(preds_j) > 0:
                from scipy.stats import mode
                pred, _ = mode(preds_j.astype(int), keepdims=False)
                final_preds.append(pred)
                valid_mask.append(j)

        return accuracy_score(y[valid_mask], final_preds)


# --- Demo ---
X_demo, y_demo = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, random_state=SEED
)
X_train, X_test, y_train, y_test = train_test_split(
    X_demo, y_demo, test_size=0.2, random_state=SEED
)

# Single decision tree
single_tree = DecisionTreeClassifier(random_state=SEED)
single_tree.fit(X_train, y_train)
single_acc = accuracy_score(y_test, single_tree.predict(X_test))

# Our bagging from scratch
bag_scratch = BaggingFromScratch(
    DecisionTreeClassifier(random_state=SEED), n_estimators=50
)
bag_scratch.fit(X_train, y_train)
bag_acc = accuracy_score(y_test, bag_scratch.predict(X_test))
oob = bag_scratch.oob_score(X_train, y_train)

print(f"Single Decision Tree accuracy:  {single_acc:.4f}")
print(f"Bagging (from scratch) accuracy: {bag_acc:.4f}")
print(f"OOB Score estimate:              {oob:.4f}")
print(f"\nImprovement: +{(bag_acc - single_acc):.4f}")

### 3.2 Sklearn BaggingClassifier & Random Forest

`BaggingClassifier` wraps any base estimator. `RandomForestClassifier` is a special case where the base estimator is a decision tree **with random feature subsets** at each split.

In [ ]:
# Sklearn BaggingClassifier
bag_sklearn = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50, oob_score=True, random_state=SEED, n_jobs=-1
)
bag_sklearn.fit(X_train, y_train)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=200, oob_score=True, random_state=SEED, n_jobs=-1
)
rf.fit(X_train, y_train)

results = pd.DataFrame({
    "Model": ["Single Tree", "Bagging (scratch)", "Bagging (sklearn)",
              "Random Forest"],
    "Test Accuracy": [
        single_acc,
        bag_acc,
        accuracy_score(y_test, bag_sklearn.predict(X_test)),
        accuracy_score(y_test, rf.predict(X_test)),
    ],
    "OOB Score": [
        np.nan, oob,
        bag_sklearn.oob_score_,
        rf.oob_score_,
    ]
})
print(results.to_string(index=False))

In [ ]:
# --- Effect of n_estimators on OOB score ---
n_range = [5, 10, 20, 50, 100, 200, 300, 500]
oob_scores = []
test_scores = []

for n in n_range:
    rf_temp = RandomForestClassifier(
        n_estimators=n, oob_score=True, random_state=SEED, n_jobs=-1
    )
    rf_temp.fit(X_train, y_train)
    oob_scores.append(rf_temp.oob_score_)
    test_scores.append(accuracy_score(y_test, rf_temp.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_range, oob_scores, "bo-", lw=2, label="OOB Score")
ax.plot(n_range, test_scores, "rs--", lw=2, label="Test Accuracy")
ax.set_xlabel("Number of Trees", fontsize=13)
ax.set_ylabel("Score", fontsize=13)
ax.set_title("Random Forest: OOB Score Tracks Test Accuracy",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

> **Key Takeaway -- Bagging:**  
> Bagging is your go-to when you have a high-variance base learner (deep trees, KNN). Random Forest adds feature randomness on top of bootstrap sampling, making trees even more diverse. The OOB score is a free validation estimate -- use it!

---

<a id="4"></a>
## 4. Boosting

**Core idea:** Train models **sequentially**. Each new model focuses on the examples the previous models got wrong.

- **AdaBoost:** Re-weight misclassified samples.
- **Gradient Boosting:** Fit new models to the **residual errors** (negative gradient of the loss).
- **XGBoost / LightGBM / CatBoost:** Highly optimized gradient boosting with regularization, histogram-based splits, and more.

### 4.1 AdaBoost Intuition

In [ ]:
# --- AdaBoost: Watch it focus on hard examples ---
X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=SEED)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for idx, n_est in enumerate([1, 3, 10, 50]):
    ada = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=n_est, random_state=SEED, algorithm="SAMME"
    )
    ada.fit(X_moons, y_moons)

    ax = axes[idx]
    xx, yy = np.meshgrid(
        np.linspace(X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5, 200),
        np.linspace(X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5, 200),
    )
    Z = ada.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="RdBu")
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="RdBu",
               edgecolors="k", s=20)
    acc = accuracy_score(y_moons, ada.predict(X_moons))
    ax.set_title(f"n_estimators={n_est}\nacc={acc:.3f}", fontsize=12)

plt.suptitle("AdaBoost: Sequential Correction of Errors",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 4.2 Gradient Boosting Step-by-Step

The gradient boosting algorithm:

1. Initialize with a constant prediction: $F_0(x) = \arg\min_\gamma \sum L(y_i, \gamma)$
2. For $m = 1$ to $M$:
   - Compute pseudo-residuals: $r_{im} = -\frac{\partial L(y_i, F_{m-1}(x_i))}{\partial F_{m-1}(x_i)}$
   - Fit a weak learner $h_m(x)$ to the pseudo-residuals
   - Update: $F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$

Where $\eta$ is the learning rate (shrinkage).

In [ ]:
# --- Gradient Boosting: Learning rate effect ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
learning_rates = [0.01, 0.1, 1.0]

for ax, lr in zip(axes, learning_rates):
    train_scores = []
    test_scores = []
    gb = GradientBoostingClassifier(
        n_estimators=300, learning_rate=lr, max_depth=3, random_state=SEED
    )
    gb.fit(X_train, y_train)

    # Staged predictions
    for i, (train_pred, test_pred) in enumerate(
        zip(gb.staged_predict(X_train), gb.staged_predict(X_test))
    ):
        train_scores.append(accuracy_score(y_train, train_pred))
        test_scores.append(accuracy_score(y_test, test_pred))

    ax.plot(train_scores, label="Train", lw=2)
    ax.plot(test_scores, label="Test", lw=2)
    ax.set_xlabel("Boosting Round")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"learning_rate = {lr}", fontsize=13, fontweight="bold")
    ax.legend()
    ax.set_ylim(0.5, 1.02)

plt.suptitle("Gradient Boosting: Effect of Learning Rate",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 4.3 XGBoost vs LightGBM vs CatBoost

| Feature | XGBoost | LightGBM | CatBoost |
|---------|---------|----------|----------|
| **Split strategy** | Level-wise | Leaf-wise | Symmetric (oblivious) trees |
| **Categorical handling** | Manual encoding needed | Built-in (optimal) | Built-in (best) |
| **Speed** | Fast | Fastest | Moderate |
| **GPU support** | Yes | Yes | Yes (excellent) |
| **Missing values** | Built-in | Built-in | Built-in |
| **Regularization** | L1, L2, gamma | L1, L2 | L2 + ordered boosting |
| **Best for** | General purpose | Large datasets, speed | Categorical-heavy data |

### 4.4 Comparison on our data

In [ ]:
import time

models_gbm = {
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        eval_metric="logloss", random_state=SEED, verbosity=0
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=SEED, verbose=-1
    ),
    "CatBoost": CatBoostClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=SEED, verbose=0
    ),
}

comparison = []
for name, model in models_gbm.items():
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    comparison.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC-ROC": roc_auc_score(y_test, y_prob),
        "Log Loss": log_loss(y_test, y_prob),
        "Train Time (s)": round(train_time, 3),
    })

df_compare = pd.DataFrame(comparison)
print(df_compare.to_string(index=False))

### 4.5 Hyperparameter Tuning with Optuna

Optuna is the gold standard for Bayesian hyperparameter optimization on Kaggle. Here is the pattern you will use in every competition:

In [ ]:
# NOTE: Optuna must be installed (pip install optuna).
# On Kaggle it is pre-installed.
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        }

        model = XGBClassifier(
            **params, eval_metric="logloss", random_state=SEED, verbosity=0
        )
        scores = cross_val_score(
            model, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1
        )
        return scores.mean()

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=30, show_progress_bar=False)

    print(f"Best AUC-ROC (CV): {study.best_value:.4f}")
    print(f"\nBest hyperparameters:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")

except ImportError:
    print("Optuna not installed. Install with: pip install optuna")
    print("Skipping hyperparameter optimization demo.")

> **Key Takeaway -- Boosting:**  
> Gradient boosting (XGBoost, LightGBM, CatBoost) is the **most powerful single-model family** for tabular data. LightGBM is fastest, CatBoost handles categoricals best, XGBoost is the most battle-tested. Always tune with Optuna or similar.

---

<a id="5"></a>
## 5. Blending & Weighted Averaging

Once you have multiple trained models, the simplest ensemble is to **average their predictions**. Better yet: find **optimal weights**.

### 5.1 Simple Averaging vs Weighted Averaging

In [ ]:
# Train diverse base models
base_models = {
    "LR": LogisticRegression(max_iter=1000, random_state=SEED),
    "RF": RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    "XGB": XGBClassifier(n_estimators=200, eval_metric="logloss",
                         random_state=SEED, verbosity=0),
    "LGBM": LGBMClassifier(n_estimators=200, random_state=SEED, verbose=-1),
    "KNN": KNeighborsClassifier(n_neighbors=15),
}

# Fit and collect predictions
preds_proba = {}
for name, model in base_models.items():
    if name == "KNN":
        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_train)
        X_te_sc = scaler.transform(X_test)
        model.fit(X_tr_sc, y_train)
        preds_proba[name] = model.predict_proba(X_te_sc)[:, 1]
    else:
        model.fit(X_train, y_train)
        preds_proba[name] = model.predict_proba(X_test)[:, 1]

# Simple average
avg_pred = np.mean(list(preds_proba.values()), axis=0)
avg_auc = roc_auc_score(y_test, avg_pred)

# Individual scores
print("Individual Model AUC-ROC:")
for name, pred in preds_proba.items():
    print(f"  {name:5s}: {roc_auc_score(y_test, pred):.4f}")
print(f"\nSimple Average: {avg_auc:.4f}")

In [ ]:
# --- Optimize blend weights with scipy ---
pred_matrix = np.column_stack(list(preds_proba.values()))
model_names = list(preds_proba.keys())

def neg_auc(weights):
    """Negative AUC for minimization."""
    weights = np.abs(weights)
    weights = weights / weights.sum()  # normalize
    blend = pred_matrix @ weights
    return -roc_auc_score(y_test, blend)

# Initial equal weights
w0 = np.ones(len(model_names)) / len(model_names)

# Optimize
result = minimize(neg_auc, w0, method="Nelder-Mead",
                  options={"maxiter": 10000})
opt_weights = np.abs(result.x)
opt_weights = opt_weights / opt_weights.sum()

opt_blend = pred_matrix @ opt_weights
opt_auc = roc_auc_score(y_test, opt_blend)

print("Optimized Weights:")
for name, w in zip(model_names, opt_weights):
    print(f"  {name:5s}: {w:.4f}")
print(f"\nOptimized Blend AUC: {opt_auc:.4f}")
print(f"Simple Average AUC:  {avg_auc:.4f}")
print(f"Improvement:         +{opt_auc - avg_auc:.4f}")

In [ ]:
# --- Rank Averaging ---
# Convert probabilities to ranks, then average ranks
from scipy.stats import rankdata

rank_preds = {}
for name, pred in preds_proba.items():
    rank_preds[name] = rankdata(pred) / len(pred)

rank_avg = np.mean(list(rank_preds.values()), axis=0)
rank_auc = roc_auc_score(y_test, rank_avg)

print(f"Simple Average AUC:    {avg_auc:.4f}")
print(f"Optimized Blend AUC:   {opt_auc:.4f}")
print(f"Rank Average AUC:      {rank_auc:.4f}")
print(f"\nRank averaging is especially useful when models produce")
print(f"predictions on very different scales (e.g., neural net vs GBM).")

In [ ]:
# --- How correlation affects ensemble quality ---
pred_df = pd.DataFrame(preds_proba)
corr = pred_df.corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Correlation heatmap
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm",
            center=0.5, ax=axes[0], square=True)
axes[0].set_title("Prediction Correlation Between Models",
                  fontsize=13, fontweight="bold")

# Ensemble AUC vs number of models (adding least correlated first)
available = list(preds_proba.keys())
selected = []
ensemble_aucs = []

# Start with the best individual model
best_model = max(preds_proba.keys(),
                 key=lambda k: roc_auc_score(y_test, preds_proba[k]))
selected.append(best_model)
available.remove(best_model)

ensemble_aucs.append(roc_auc_score(y_test, preds_proba[best_model]))

while available:
    # Pick model with lowest avg correlation to selected models
    best_next = min(available,
                    key=lambda k: np.mean([corr.loc[k, s] for s in selected]))
    selected.append(best_next)
    available.remove(best_next)

    blend = np.mean([preds_proba[s] for s in selected], axis=0)
    ensemble_aucs.append(roc_auc_score(y_test, blend))

axes[1].plot(range(1, len(selected) + 1), ensemble_aucs, "bo-", lw=2, markersize=8)
for i, name in enumerate(selected):
    axes[1].annotate(name, (i + 1, ensemble_aucs[i]),
                     textcoords="offset points", xytext=(0, 10),
                     fontsize=10, ha="center")
axes[1].set_xlabel("Number of Models in Ensemble", fontsize=12)
axes[1].set_ylabel("AUC-ROC", fontsize=12)
axes[1].set_title("Greedy Forward Selection (Least Correlated First)",
                  fontsize=13, fontweight="bold")
axes[1].set_xticks(range(1, len(selected) + 1))

plt.tight_layout()
plt.show()

> **Key Takeaway -- Blending:**  
> Simple averaging is a strong baseline. Optimized weights squeeze out extra performance. **Rank averaging** is robust when model scales differ. The key insight: **diversity matters more than individual accuracy** -- adding a weaker but uncorrelated model often helps more than adding a stronger but correlated one.

---

<a id="6"></a>
## 6. Stacking

**The most powerful ensemble technique in competitions.**

Stacking trains a **meta-learner** on out-of-fold (OOF) predictions from base models. The key challenge is avoiding data leakage.

### 6.1 K-Fold Stacking from Scratch (The Right Way)

```
For each fold k:
    1. Train base models on folds != k
    2. Predict on fold k (these are OOF predictions)
    3. Predict on test set
Average test predictions across folds
Train meta-learner on OOF predictions
```

In [ ]:
def get_oof_predictions(models, X_train, y_train, X_test, n_folds=5):
    """
    Generate out-of-fold predictions for stacking.
    This is the CORRECT way -- no data leakage!
    """
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    n_train = X_train.shape[0]
    n_test = X_test.shape[0]

    # Storage
    oof_train = np.zeros((n_train, len(models)))  # OOF predictions
    oof_test = np.zeros((n_test, len(models)))     # Averaged test predictions

    for model_idx, (name, model_template) in enumerate(models.items()):
        print(f"  Training {name}...")
        test_preds_folds = np.zeros((n_test, n_folds))

        for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]

            from sklearn.base import clone
            model = clone(model_template)

            # Handle KNN scaling
            if name == "KNN":
                scaler = StandardScaler()
                X_tr = scaler.fit_transform(X_tr)
                X_val = scaler.transform(X_val)
                X_te = scaler.transform(X_test)
            else:
                X_te = X_test

            model.fit(X_tr, y_tr)

            # OOF predictions (probabilities)
            oof_train[val_idx, model_idx] = model.predict_proba(X_val)[:, 1]
            test_preds_folds[:, fold_idx] = model.predict_proba(X_te)[:, 1]

        # Average test predictions across folds
        oof_test[:, model_idx] = test_preds_folds.mean(axis=1)

        # OOF score
        oof_auc = roc_auc_score(y_train, oof_train[:, model_idx])
        print(f"    OOF AUC: {oof_auc:.4f}")

    return oof_train, oof_test


# Define base models for stacking
stack_base_models = {
    "LR": LogisticRegression(max_iter=1000, random_state=SEED),
    "RF": RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    "XGB": XGBClassifier(n_estimators=200, eval_metric="logloss",
                         random_state=SEED, verbosity=0),
    "LGBM": LGBMClassifier(n_estimators=200, random_state=SEED, verbose=-1),
    "KNN": KNeighborsClassifier(n_neighbors=15),
}

print("Generating OOF predictions...")
oof_train, oof_test = get_oof_predictions(
    stack_base_models, X_train, y_train, X_test, n_folds=5
)
print("\nOOF train shape:", oof_train.shape)
print("OOF test shape: ", oof_test.shape)

In [ ]:
# --- Train meta-learner ---
print("=" * 50)
print("META-LEARNER COMPARISON")
print("=" * 50)

meta_learners = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Ridge": RidgeClassifier(random_state=SEED),
    "LightGBM (meta)": LGBMClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.05,
        random_state=SEED, verbose=-1
    ),
}

for name, meta in meta_learners.items():
    meta.fit(oof_train, y_train)
    if hasattr(meta, "predict_proba"):
        meta_pred = meta.predict_proba(oof_test)[:, 1]
        meta_auc = roc_auc_score(y_test, meta_pred)
    else:
        meta_pred = meta.decision_function(oof_test)
        meta_auc = roc_auc_score(y_test, meta_pred)
    meta_acc = accuracy_score(y_test, meta.predict(oof_test))
    print(f"\n{name}:")
    print(f"  AUC-ROC:  {meta_auc:.4f}")
    print(f"  Accuracy: {meta_acc:.4f}")

# Compare with simple average of OOF test predictions
simple_avg = oof_test.mean(axis=1)
print(f"\nSimple Average Blend:")
print(f"  AUC-ROC:  {roc_auc_score(y_test, simple_avg):.4f}")

### 6.2 Multi-Level Stacking

In competitions, top solutions often use 2-3 levels of stacking:

```
Level 0: Raw features
    -> Diverse base models (LR, RF, XGB, LGBM, KNN, SVM, NN)
    -> Generate OOF predictions

Level 1: OOF predictions from Level 0 (+ optionally original features)
    -> Second set of models (usually simpler)
    -> Generate OOF predictions

Level 2: OOF predictions from Level 1
    -> Simple meta-learner (logistic regression, simple average)
    -> Final prediction
```

**Warning:** Each level introduces complexity and overfitting risk. In practice, 2 levels is usually the sweet spot. Beyond 2, you often see diminishing returns.

### 6.3 Sklearn's StackingClassifier

For convenience, sklearn provides a built-in stacking implementation:

In [ ]:
# Sklearn StackingClassifier
sklearn_stack = StackingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000, random_state=SEED)),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=SEED)),
        ("xgb", XGBClassifier(n_estimators=100, eval_metric="logloss",
                              random_state=SEED, verbosity=0)),
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5, n_jobs=-1, passthrough=False
)

sklearn_stack.fit(X_train, y_train)
sk_pred = sklearn_stack.predict_proba(X_test)[:, 1]
print(f"Sklearn StackingClassifier AUC: {roc_auc_score(y_test, sk_pred):.4f}")
print(f"\nNote: Our from-scratch version gives more control (e.g., saving OOF preds).")

> **Key Takeaway -- Stacking:**  
> K-fold stacking is the gold standard for competitions. The critical detail is generating OOF predictions to avoid leakage. Use a simple meta-learner (logistic regression) to avoid second-level overfitting. Diversity in base models is more important than individual model quality.

---

<a id="7"></a>
## 7. Advanced: Snapshot Ensembles

**Idea:** During training with cyclic learning rate schedules (cosine annealing), the model passes through multiple local minima. Save a "snapshot" at each minimum, then ensemble these snapshots -- **free ensemble from a single training run!**

Reference: [Huang et al., 2017 - "Snapshot Ensembles: Train 1, Get M for Free"](https://arxiv.org/abs/1704.00109)

### Cosine Annealing Schedule

$$\alpha(t) = \frac{\alpha_0}{2}\left(\cos\left(\frac{\pi \cdot \text{mod}(t-1, \lceil T/M \rceil)}{\lceil T/M \rceil}\right) + 1\right)$$

Where $T$ is total epochs, $M$ is number of cycles.

In [ ]:
# --- Cosine Annealing with Warm Restarts ---
def cosine_annealing_schedule(t, T, M, alpha_0=0.1):
    """Cosine annealing with M restarts over T total steps."""
    cycle_length = T // M
    t_in_cycle = t % cycle_length
    return (alpha_0 / 2) * (np.cos(np.pi * t_in_cycle / cycle_length) + 1)

T = 300  # total steps
M_cycles = 5  # number of restart cycles

t_values = np.arange(T)
lr_values = [cosine_annealing_schedule(t, T, M_cycles) for t in t_values]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t_values, lr_values, "b-", lw=1.5)

# Mark snapshot points (cycle minima)
cycle_length = T // M_cycles
snapshot_points = [(i + 1) * cycle_length - 1 for i in range(M_cycles)]
for sp in snapshot_points:
    ax.axvline(x=sp, color="red", ls="--", alpha=0.7)
    ax.plot(sp, lr_values[sp], "r*", markersize=15)

ax.set_xlabel("Training Step", fontsize=12)
ax.set_ylabel("Learning Rate", fontsize=12)
ax.set_title("Cosine Annealing with Warm Restarts -- Stars = Snapshot Points",
             fontsize=14, fontweight="bold")
ax.legend(["Learning Rate", "Snapshot (save model)"], fontsize=11)
plt.tight_layout()
plt.show()

print(f"With {M_cycles} cycles, we get {M_cycles} diverse models for FREE!")
print("Each snapshot explores a different region of the loss landscape.")

In [ ]:
# --- Simulate snapshot ensembles with sklearn ---
# We simulate by training RF with different random seeds (analogous to
# different local minima in the loss landscape)

snapshot_models = []
snapshot_preds = []

for i in range(5):
    # Different subsample = different "snapshot"
    model = RandomForestClassifier(
        n_estimators=50,
        max_features=0.6 + 0.08 * i,  # slight variation
        max_samples=0.7 + 0.06 * i,
        random_state=SEED + i * 17,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    pred = model.predict_proba(X_test)[:, 1]
    snapshot_models.append(model)
    snapshot_preds.append(pred)
    auc = roc_auc_score(y_test, pred)
    print(f"Snapshot {i+1} AUC: {auc:.4f}")

# Ensemble all snapshots
ensemble_pred = np.mean(snapshot_preds, axis=0)
ensemble_auc = roc_auc_score(y_test, ensemble_pred)
print(f"\nSnapshot Ensemble AUC: {ensemble_auc:.4f}")
print("Improvement over best single snapshot: "
      f"+{ensemble_auc - max(roc_auc_score(y_test, p) for p in snapshot_preds):.4f}")

<a id="8"></a>
## 8. Advanced: Knowledge Distillation

**Problem:** Your competition ensemble is amazing but too slow for production.  
**Solution:** Train a smaller "student" model to mimic the ensemble "teacher" using **soft labels**.

### The Key Insight: Soft Labels

Hard labels: `[0, 1, 0]` -- only the correct class has probability 1.  
Soft labels: `[0.05, 0.85, 0.10]` -- the teacher's probability distribution.

Soft labels contain **"dark knowledge"** -- they encode inter-class similarities that hard labels miss. A cat being 10% dog and 2% car is informative!

### Temperature Scaling

$$q_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

Higher temperature $T$ produces softer probabilities, revealing more structure.

In [ ]:
# --- Knowledge Distillation Demo ---

# Step 1: Create a powerful "teacher" ensemble
teacher_models = [
    RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    XGBClassifier(n_estimators=300, eval_metric="logloss",
                  random_state=SEED, verbosity=0),
    LGBMClassifier(n_estimators=300, random_state=SEED, verbose=-1),
]

# Fit teachers
teacher_preds = []
for m in teacher_models:
    m.fit(X_train, y_train)
    teacher_preds.append(m.predict_proba(X_train)[:, 1])

# Soft labels from teacher ensemble (average probabilities)
soft_labels = np.mean(teacher_preds, axis=0)

# Temperature scaling (soften the distribution)
def temperature_scale(probs, T=2.0):
    """Apply temperature scaling to probabilities."""
    # Convert prob to logit, scale, convert back
    logits = np.log(probs / (1 - probs + 1e-10) + 1e-10)
    scaled_logits = logits / T
    return 1 / (1 + np.exp(-scaled_logits))

temperatures = [0.5, 1.0, 2.0, 5.0]
fig, axes = plt.subplots(1, len(temperatures), figsize=(20, 4))

for ax, T_val in zip(axes, temperatures):
    scaled = temperature_scale(soft_labels, T_val)
    ax.hist(scaled, bins=50, alpha=0.7, color="steelblue", edgecolor="white")
    ax.set_title(f"T = {T_val}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Soft Label Value")
    ax.set_xlim(0, 1)

plt.suptitle("Effect of Temperature on Soft Label Distribution",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Step 2: Train student on soft labels
# Student trained on hard labels
student_hard = DecisionTreeClassifier(max_depth=5, random_state=SEED)
student_hard.fit(X_train, y_train)
hard_auc = roc_auc_score(y_test, student_hard.predict_proba(X_test)[:, 1])

# Student trained on soft labels (using teacher's predictions as targets)
soft_targets = (soft_labels > 0.5).astype(int)
student_soft = DecisionTreeClassifier(max_depth=5, random_state=SEED)
student_soft.fit(X_train, soft_targets)
soft_auc = roc_auc_score(y_test, student_soft.predict_proba(X_test)[:, 1])

# Teacher ensemble on test
teacher_test_preds = np.mean([m.predict_proba(X_test)[:, 1]
                              for m in teacher_models], axis=0)
teacher_auc = roc_auc_score(y_test, teacher_test_preds)

print(f"Teacher Ensemble AUC:            {teacher_auc:.4f}")
print(f"Student (hard labels) AUC:       {hard_auc:.4f}")
print(f"Student (soft/distilled) AUC:    {soft_auc:.4f}")
print(f"\nThe student trained on soft labels benefits from the teacher's knowledge!")

> **Key Takeaway -- Advanced Techniques:**  
> Snapshot ensembles give you multiple models for the cost of one training run -- use cyclic learning rates. Knowledge distillation compresses an ensemble into a deployable single model using soft labels. Both are widely used in competition solutions and production systems.

---

<a id="9"></a>
## 9. Practical Competition Pipeline

Here is the **end-to-end pipeline** you should follow in every Kaggle competition. We demonstrate with synthetic data, but the pattern is universal.

### Pipeline Overview

```
1. Load & preprocess data
2. Train diverse base models with K-fold
3. Generate OOF predictions (stacking features)
4. Optimize blend weights on OOF predictions
5. Train meta-learner
6. Generate final submission
```

In [ ]:
# ============================================================
# FULL COMPETITION PIPELINE
# ============================================================

# --- Step 1: Generate competition-like data ---
X_full, y_full = make_classification(
    n_samples=5000, n_features=30, n_informative=15,
    n_redundant=8, n_clusters_per_class=3,
    flip_y=0.05, random_state=SEED
)

# Simulate train/test split (in competition, test has no labels)
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=SEED
)
print(f"Train: {X_train_full.shape}, Test: {X_test_full.shape}")

# --- Step 2: Define diverse base models ---
pipeline_models = {
    "LR": LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
    "RF": RandomForestClassifier(n_estimators=300, max_depth=10,
                                  random_state=SEED, n_jobs=-1),
    "XGB": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         eval_metric="logloss", random_state=SEED, verbosity=0),
    "LGBM": LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8,
                            random_state=SEED, verbose=-1),
    "CatBoost": CatBoostClassifier(n_estimators=300, max_depth=6,
                                    learning_rate=0.05, random_state=SEED,
                                    verbose=0),
    "MLP": MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500,
                          random_state=SEED),
}

# --- Step 3: Generate OOF predictions ---
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_preds = np.zeros((X_train_full.shape[0], len(pipeline_models)))
test_preds = np.zeros((X_test_full.shape[0], len(pipeline_models)))

for model_idx, (name, model_template) in enumerate(pipeline_models.items()):
    print(f"\n--- {name} ---")
    fold_test_preds = np.zeros((X_test_full.shape[0], N_FOLDS))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):
        X_tr = X_train_full[tr_idx]
        y_tr = y_train_full[tr_idx]
        X_val = X_train_full[val_idx]

        from sklearn.base import clone
        model = clone(model_template)

        # Scale for MLP
        if name == "MLP":
            sc = StandardScaler()
            X_tr = sc.fit_transform(X_tr)
            X_val = sc.transform(X_val)
            X_te = sc.transform(X_test_full)
        else:
            X_te = X_test_full

        model.fit(X_tr, y_tr)

        oof_preds[val_idx, model_idx] = model.predict_proba(X_val)[:, 1]
        fold_test_preds[:, fold] = model.predict_proba(X_te)[:, 1]

    test_preds[:, model_idx] = fold_test_preds.mean(axis=1)

    oof_auc = roc_auc_score(y_train_full, oof_preds[:, model_idx])
    print(f"  OOF AUC: {oof_auc:.4f}")

print("\n" + "=" * 50)

In [ ]:
# --- Step 4: Optimize blend weights on OOF predictions ---
model_names_pipe = list(pipeline_models.keys())

def neg_auc_oof(weights):
    w = np.abs(weights)
    w = w / w.sum()
    blend = oof_preds @ w
    return -roc_auc_score(y_train_full, blend)

w0 = np.ones(len(model_names_pipe)) / len(model_names_pipe)
result = minimize(neg_auc_oof, w0, method="Nelder-Mead",
                  options={"maxiter": 10000})
opt_w = np.abs(result.x)
opt_w = opt_w / opt_w.sum()

print("Optimized Blend Weights (on OOF):")
for name, w in zip(model_names_pipe, opt_w):
    print(f"  {name:10s}: {w:.4f}")

# --- Step 5: Create final submission ---
final_blend = test_preds @ opt_w
final_auc = roc_auc_score(y_test_full, final_blend)

# Also try stacking
meta = LogisticRegression(max_iter=1000, random_state=SEED)
meta.fit(oof_preds, y_train_full)
stacked_pred = meta.predict_proba(test_preds)[:, 1]
stacked_auc = roc_auc_score(y_test_full, stacked_pred)

# Simple average baseline
simple_blend = test_preds.mean(axis=1)
simple_auc = roc_auc_score(y_test_full, simple_blend)

print(f"\n{'='*50}")
print(f"FINAL RESULTS")
print(f"{'='*50}")
print(f"Simple Average:       {simple_auc:.4f}")
print(f"Optimized Blend:      {final_auc:.4f}")
print(f"Stacking (LR meta):   {stacked_auc:.4f}")

# Create submission DataFrame (simulating competition)
submission = pd.DataFrame({
    "id": np.arange(len(y_test_full)),
    "target": final_blend
})
print(f"\nSubmission shape: {submission.shape}")
print(submission.head())

> **Key Takeaway -- Pipeline:**  
> This is the exact pipeline used by competition winners. The key steps are: (1) diverse base models, (2) proper K-fold OOF predictions, (3) weight optimization on OOF -- NOT on test data, (4) simple meta-learner to avoid overfitting. **Always validate ensembling decisions on OOF scores, never on the leaderboard.**

---

<a id="10"></a>
## 10. Diversity Analysis

Understanding **why** ensembles work requires analyzing model diversity. Two models that always agree add no value to each other. The ideal ensemble has models that are individually strong but make **different errors**.

### 10.1 Correlation Heatmap & Diversity Metrics

In [ ]:
# --- Comprehensive Diversity Analysis ---
oof_df = pd.DataFrame(oof_preds, columns=model_names_pipe)

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 2, hspace=0.3, wspace=0.3)

# 1. Correlation heatmap
ax1 = fig.add_subplot(gs[0, 0])
corr_matrix = oof_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="RdYlGn_r",
            center=0.7, ax=ax1, square=True, mask=mask,
            vmin=0.5, vmax=1.0)
ax1.set_title("Prediction Correlation (OOF)", fontsize=13, fontweight="bold")

# 2. Pairwise disagreement rate
ax2 = fig.add_subplot(gs[0, 1])
binary_preds = (oof_preds > 0.5).astype(int)
n_models = binary_preds.shape[1]
disagreement = np.zeros((n_models, n_models))
for i in range(n_models):
    for j in range(n_models):
        disagreement[i, j] = np.mean(binary_preds[:, i] != binary_preds[:, j])

sns.heatmap(pd.DataFrame(disagreement, index=model_names_pipe,
                          columns=model_names_pipe),
            annot=True, fmt=".3f", cmap="YlOrRd", ax=ax2, square=True)
ax2.set_title("Pairwise Disagreement Rate", fontsize=13, fontweight="bold")

# 3. Individual vs Ensemble performance
ax3 = fig.add_subplot(gs[1, 0])
individual_aucs = [roc_auc_score(y_train_full, oof_preds[:, i])
                   for i in range(n_models)]
colors = plt.cm.Set2(np.linspace(0, 1, n_models))
bars = ax3.barh(model_names_pipe, individual_aucs, color=colors, edgecolor="gray")
oof_ensemble_auc = roc_auc_score(y_train_full, oof_preds.mean(axis=1))
ax3.axvline(x=oof_ensemble_auc, color="red", lw=2, ls="--",
            label=f"Ensemble: {oof_ensemble_auc:.4f}")
ax3.set_xlabel("OOF AUC-ROC", fontsize=12)
ax3.set_title("Individual vs Ensemble (OOF)", fontsize=13, fontweight="bold")
ax3.legend(fontsize=11)
ax3.set_xlim(min(individual_aucs) - 0.02, max(max(individual_aucs),
             oof_ensemble_auc) + 0.01)

# 4. Cumulative ensemble: adding models one by one
ax4 = fig.add_subplot(gs[1, 1])
sorted_indices = np.argsort(individual_aucs)[::-1]
cumulative_aucs = []
for k in range(1, n_models + 1):
    selected = sorted_indices[:k]
    blend = oof_preds[:, selected].mean(axis=1)
    cumulative_aucs.append(roc_auc_score(y_train_full, blend))

ax4.plot(range(1, n_models + 1), cumulative_aucs, "bo-", lw=2, markersize=8)
for k in range(n_models):
    ax4.annotate(model_names_pipe[sorted_indices[k]],
                 (k + 1, cumulative_aucs[k]),
                 textcoords="offset points", xytext=(5, 10),
                 fontsize=9, rotation=15)
ax4.set_xlabel("Number of Models Added", fontsize=12)
ax4.set_ylabel("Ensemble AUC-ROC (OOF)", fontsize=12)
ax4.set_title("Cumulative Ensemble (Best First)", fontsize=13, fontweight="bold")
ax4.set_xticks(range(1, n_models + 1))

plt.suptitle("Diversity Analysis Dashboard", fontsize=16, fontweight="bold", y=1.01)
plt.show()

In [ ]:
# --- When does adding a model help vs hurt? ---
print("ANALYSIS: When Adding Models Helps vs Hurts")
print("=" * 55)
print()

# For each model, compute the ensemble AUC with and without it
full_ensemble_auc = roc_auc_score(y_train_full, oof_preds.mean(axis=1))

print(f"Full Ensemble AUC (all {n_models} models): {full_ensemble_auc:.4f}")
print()

for i, name in enumerate(model_names_pipe):
    # Leave-one-out ensemble
    mask = np.ones(n_models, dtype=bool)
    mask[i] = False
    loo_pred = oof_preds[:, mask].mean(axis=1)
    loo_auc = roc_auc_score(y_train_full, loo_pred)

    delta = full_ensemble_auc - loo_auc
    direction = "HELPS" if delta > 0 else "HURTS"
    symbol = "+" if delta > 0 else ""

    # Average correlation with other models
    avg_corr = corr_matrix.loc[name].drop(name).mean()

    print(f"  {name:10s}  | Individual AUC: {individual_aucs[i]:.4f} | "
          f"Avg Corr: {avg_corr:.3f} | "
          f"Removing it: {symbol}{delta:.4f} ({direction})")

print()
print("Models with LOW correlation to others contribute MORE to the ensemble.")
print("A weaker but diverse model often beats a stronger correlated one.")

> **Key Takeaway -- Diversity:**  
> The value of a model in an ensemble is determined by both its individual accuracy AND its diversity (low correlation with other models). Always analyze the correlation structure before deciding which models to include. Remove models that are highly correlated -- they add complexity without improving performance.

---

<a id="11"></a>
## 11. Further Reading

### Papers
- Breiman (1996) - [Bagging Predictors](https://link.springer.com/article/10.1023/A:1018054314350)
- Freund & Schapire (1997) - [A Decision-Theoretic Generalization of On-Line Learning (AdaBoost)](https://www.sciencedirect.com/science/article/pii/S002200009791504X)
- Wolpert (1992) - [Stacked Generalization](https://www.sciencedirect.com/science/article/abs/pii/S0893608005800231)
- Chen & Guestrin (2016) - [XGBoost: A Scalable Tree Boosting System](https://arxiv.org/abs/1603.02754)
- Ke et al. (2017) - [LightGBM: A Highly Efficient Gradient Boosting Decision Tree](https://papers.nips.cc/paper/6907-lightgbm-a-highly-efficient-gradient-boosting-decision-tree)
- Huang et al. (2017) - [Snapshot Ensembles: Train 1, Get M for Free](https://arxiv.org/abs/1704.00109)
- Hinton et al. (2015) - [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531)

### Kaggle Resources
- [Kaggle Ensembling Guide](https://mlwave.com/kaggle-ensembling-guide/) by MLWave
- [Introduction to Ensembling/Stacking](https://www.kaggle.com/arthurtok/introduction-to-ensembling-stacking-in-python) by Anisotropic
- [Stacking Made Easy](https://www.kaggle.com/serigne/stacked-regressions-top-4-on-leaderboard) by Serigne

### Competition Write-ups
- Nearly every top competition solution on [Kaggle Solutions](https://farid.one/kaggle-solutions/) features ensembling

---

## Summary Cheat Sheet

| Method | Reduces | Key Trick | When to Use |
|--------|---------|-----------|-------------|
| **Bagging** | Variance | Bootstrap samples + averaging | High-variance models (trees) |
| **Random Forest** | Variance | Bagging + random feature subsets | Default first try for tabular |
| **AdaBoost** | Bias | Re-weight misclassified samples | Simple base learners (stumps) |
| **Gradient Boosting** | Bias + Variance | Fit residuals sequentially | Almost always for tabular |
| **XGBoost** | Bias + Variance | Regularized GBM + system optimization | Competition workhorse |
| **LightGBM** | Bias + Variance | Leaf-wise growth, histograms | Large datasets, speed |
| **CatBoost** | Bias + Variance | Ordered boosting, native categoricals | Categorical-heavy data |
| **Simple Average** | Variance | Average predictions | Quick baseline ensemble |
| **Weighted Average** | Variance | Optimize weights on OOF | Better than simple average |
| **Rank Average** | Variance | Average ranks, not probabilities | Different-scale predictions |
| **Stacking** | Both | Meta-learner on OOF predictions | Competition gold standard |
| **Snapshot Ensemble** | Variance | Cyclic LR, save at minima | Single training budget |
| **Distillation** | Complexity | Soft labels from teacher | Deployment / inference speed |

---

## Thank You!

If you made it this far, you now have a **complete toolkit** for ensemble methods in competitive ML. These techniques are responsible for virtually every top Kaggle finish.

### Quick Action Items:
1. **Start simple:** Average 2-3 diverse GBMs (XGBoost + LightGBM + CatBoost)
2. **Add stacking** when you have a solid base of models
3. **Always check diversity** -- correlation heatmaps are your friend
4. **Use OOF scores** to make all ensembling decisions, never the leaderboard

---

<center>

### If you found this notebook helpful, please give it an upvote!

It helps the community discover useful content and motivates me to create more educational resources.

**Happy ensembling, and good luck in your next competition!**

</center>